## Start GeoFunctions using Spark Esri.

In [ ]:
import spark_esri as se  # Make sure this is the first import.

In [ ]:
import geofunctions as S
import pyspark.sql.functions as F

### Create a spark instance.

In [ ]:
extra_java_options = "-XX:+UseCompressedOops -XX:+AggressiveHeap"

config = {
    "spark.driver.memory": "31G",
    "spark.executor.memory": "31G",
    "spark.driver.extraJavaOptions": extra_java_options,
    "spark.executor.extraJavaOptions": extra_java_options,
    "spark.kryo.unsafe": True,
    "spark.memory.offHeap.enable": True,
    "spark.memory.offHeap.size": "31G",
    "spark.jars": "geofunctions-0.6.jar",
}

spark = se.spark_start(config=config)

In [ ]:
spark.version

### Register the spark SQL functions.

In [ ]:
S.st_register_functions()

### Test the dataframe functions.

In [ ]:
cell = 100.0

In [ ]:
(
    spark
    # Generate 10 rows.
    .range(10)
    .withColumn("lon", F.rand() * 360 - 180)
    .withColumn("lat", F.rand() * 180 - 90)
    .withColumn("q", S.st_lontoq("lon", cell))
    .withColumn("r", S.st_lattor("lat", cell))
    .show(truncate=False)
)

### Test the SQL functions.

In [ ]:
spark.sql(
    f"""
with t as (
select id,rand() * 360 - 180 lon,rand() * 180 - 90 lat
from range(10)
)

select *,st_lontoq(lon,{cell}D) q, st_lattor(lat,{cell}D) r from t
"""
).show(truncate=False)